# Setup SLM & Ngrok Tunnel trên Kaggle (Bằng Transformers gốc)
Notebook này được thiết kế để chạy Qwen3 trên GPU của Kaggle sử dụng thư viện `transformers` nổi tiếng (ổn định tuyệt đối, không sợ kén phần cứng như `vllm`). Nó bọc mô hình bằng FastAPI và tạo Public URL vượt tường lửa qua Ngrok.

**Lưu ý quan trọng**: Đảm bảo bật **GPU (T4x2)** trên Kaggle.

In [ ]:
!pip install -q pyngrok fastapi uvicorn transformers accelerate nest_asyncio

In [ ]:
import os
from pyngrok import ngrok
import nest_asyncio

# LẤY TOKEN NGROK CỦA BẠN TỪ TRANG CHỦ NGROK.COM
NGROK_AUTH_TOKEN = "ĐIỀN_TOKEN_NGROK_CỦA_BẠN_VÀO_ĐÂY"

# Cấp quyền cho tunnel Ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Bật bypass loop cho FastAPI chạy mượt mà trong Jupyter Notebook
nest_asyncio.apply()

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
import torch

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

print(f"⏳ Đang tải siêu mô hình {MODEL_NAME} bằng thuật toán cốt lõi Transformers. Chờ xíu nhé...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# device_map="auto" tự động phân tích và gắn model dàn đều lên cả 2 card T4 của Kaggle siêu an toàn!
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)
print("✅ Qwen3-4B đã lên nòng (Giao diện Transformers)!")

app = FastAPI()

@app.post("/v1/generate")
async def generate(request: Request):
    data = await request.json()
    prompt = data.get("prompt", "")
    
    # Dùng chat template chuẩn hệ thống của Qwen3 để giữ mạch logic cao nhất
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # Khóa gradient và bắt đầu sinh chữ
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=1024,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
        
    # Trích xuất phần text sinh ra sau phần prompt bị lặp
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
    
    return JSONResponse(content={"text": generated_text})

In [ ]:
# Chặn rác đường hầm trước khi chạy mới
ngrok.kill()

# Đào đường hầm Ngrok tới cổng 8000 của FastAPI
public_url = ngrok.connect(8000).public_url

print("=" * 75)
print(f"🔥 ĐÂY LÀ URL PUBLIC CỦA BẠN: {public_url}")
print("👉 SAO CHÉP URL TRÊN VÀ DÁN VÀO BIẾN SLM_NGROK_URL TRONG FILE .env CỦA BACKEND!")
print("=" * 75)

# Hoạt động web server để hứng requests 
uvicorn.run(app, host="0.0.0.0", port=8000)